[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/powerzoojax/PowerZooPy/blob/main/docs/en/examples/notebooks/nb02_sb3_plugin.ipynb)

# NB02 — Plug in SB3

> **Prerequisites**: [NB01 — Quickstart](./nb01_quickstart.ipynb)  
> **Time**: ~10 minutes

## What You'll Learn

1. The four wrapper layers and which one to use for your library
2. Connect **Stable-Baselines3 SAC** to Task 2 (battery storage arbitrage) in under 10 lines
3. Plot a learning curve and compare to the RandomPolicy baseline

## Setup

In [ ]:
# Uncomment to install in Colab or a fresh environment
# !pip install powerzoo stable-baselines3 -q

import powerzoo
print(f"PowerZoo version: {powerzoo.__version__}")

In [ ]:
from powerzoo.tasks import make_task_env
from powerzoo.wrappers import GymnasiumWrapper, MARLWrapper
from powerzoo.wrappers.flatten import FlattenWrapper
from powerzoo.benchmarks.policies import RandomPolicy
from powerzoo.benchmarks import evaluate

import numpy as np
import matplotlib.pyplot as plt

## 1. The Four Wrapper Layers

PowerZoo environments have a multi-agent native interface. Different RL libraries
expect different interfaces. Three wrappers cover all common cases:

In [ ]:
task_name = "marl_der_arbitrage"   # Task 2: 3-battery storage arbitrage

# Layer 0: PowerZoo native (PettingZoo Parallel API)
env_raw  = make_task_env(task_name, split="train")

# Layer 1: Gymnasium-compatible single-agent env
#   - obs: flat np.ndarray  |  action: flat np.ndarray
#   - For: evaluate(), quick prototyping
env_gym  = GymnasiumWrapper(env_raw)

# Layer 2: Flatten dict obs/act → 1D vectors  (required for SB3 / CleanRL)
env_flat = FlattenWrapper(env_gym)

# Layer 3: PettingZoo Parallel API (for RLlib / MAPPO)
env_marl = MARLWrapper(make_task_env(task_name, split="train"))

print("env_raw  (native):", type(env_raw).__name__)
print("env_gym  (Gym):   ", type(env_gym).__name__,
      "| obs:", env_gym.observation_space.shape)
print("env_flat (flat):  ", type(env_flat).__name__,
      "| obs:", env_flat.observation_space.shape)
print("env_marl (MARL):  ", type(env_marl).__name__,
      "| agents:", env_marl.possible_agents)

### Wrapper routing summary

```
make_task_env()
     │
     ├─► GymnasiumWrapper  ──► FlattenWrapper  ──► SB3 / CleanRL / Tianshou
     │        (gym.Env)         (flat obs/act)
     │
     └─► MARLWrapper  ──────────────────────────► RLlib / MAPPO / IPPO
              (PettingZoo Parallel API)
```

For **single-agent** algorithms: use `FlattenWrapper(GymnasiumWrapper(env))`  
For **multi-agent** algorithms: use `MARLWrapper(env)` directly

## 2. Baseline: RandomPolicy Score

Before training, measure the random policy score — this is the `0` anchor.

In [ ]:
eval_env = FlattenWrapper(GymnasiumWrapper(make_task_env(task_name, split="test")))
random_policy = RandomPolicy(eval_env.action_space, seed=0)

random_result = evaluate(
    random_policy, eval_env,
    n_episodes=5,
    task_id=task_name,
    verbose=True,
)
print(f"\nRandom policy mean reward: {random_result['mean_reward']:.2f}")

## 3. Train SAC with Stable-Baselines3

We use **Task 2** here because it's the most natural single-agent RL problem:
one battery, continuous charge/discharge action, SOC state, price signal.

> The `marl_der_arbitrage` task has 3 batteries by default.  
> `GymnasiumWrapper` combines all agents into one flat state/action, so
> from SB3's perspective it is a standard single-agent continuous control task.

In [ ]:
try:
    from stable_baselines3 import SAC
    from stable_baselines3.common.callbacks import EvalCallback
    from stable_baselines3.common.monitor import Monitor
    SB3_AVAILABLE = True
except ImportError:
    print("stable-baselines3 not installed — skipping SAC training.")
    print("Install with: pip install stable-baselines3")
    SB3_AVAILABLE = False

In [ ]:
if SB3_AVAILABLE:
    # Training environment — Task 2 train split
    train_env = Monitor(
        FlattenWrapper(GymnasiumWrapper(make_task_env(task_name, split="train")))
    )

    # SAC is off-policy and well-suited for continuous-action tasks
    model = SAC(
        "MlpPolicy",
        train_env,
        learning_rate=3e-4,
        buffer_size=50_000,
        batch_size=256,
        learning_starts=1_000,
        verbose=0,
        seed=42,
    )

    print(f"Policy network:  {model.policy}")
    print(f"Obs dim:  {train_env.observation_space.shape}")
    print(f"Act dim:  {train_env.action_space.shape}")

In [ ]:
if SB3_AVAILABLE:
    # Train for 50k steps (~1 minute on CPU)
    # Increase to 500k for more meaningful results
    TOTAL_STEPS = 50_000
    EVAL_FREQ   = 5_000

    episode_rewards = []
    eval_steps = []

    class RewardLogCallback(EvalCallback.__bases__[0]):
        """Lightweight callback to track episode rewards during training."""
        def _on_step(self):
            for info in self.locals.get('infos', []):
                if 'episode' in info:
                    episode_rewards.append(info['episode']['r'])
                    eval_steps.append(self.num_timesteps)
            return True

    model.learn(
        total_timesteps=TOTAL_STEPS,
        callback=RewardLogCallback(),
        progress_bar=False,
    )
    print(f"Training done. Last episode reward: {episode_rewards[-1]:.2f}")

## 4. Plot Learning Curve

In [ ]:
if SB3_AVAILABLE and episode_rewards:
    # Smooth with a rolling window
    window = max(1, len(episode_rewards) // 20)
    smoothed = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(episode_rewards, alpha=0.3, color='steelblue', linewidth=0.8, label='Raw')
    ax.plot(smoothed, color='steelblue', linewidth=2.0, label=f'Smoothed (w={window})')
    ax.axhline(random_result['mean_reward'], color='tomato', linewidth=1.5,
               linestyle='--', label=f'Random baseline ({random_result["mean_reward"]:.0f})')
    ax.set_xlabel('Episode')
    ax.set_ylabel('Episode reward')
    ax.set_title(f'SAC on {task_name} — training curve')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("(SB3 not available — skipping plot)")

## 5. Evaluate the Trained SAC Policy

In [ ]:
if SB3_AVAILABLE:
    class SB3Policy:
        """Thin wrapper so SB3 model satisfies the act(obs, info) interface."""
        def __init__(self, model):
            self.model = model

        def act(self, obs, info=None):
            action, _ = self.model.predict(obs, deterministic=True)
            return action

    sac_policy = SB3Policy(model)

    sac_result = evaluate(
        sac_policy, eval_env,
        n_episodes=5,
        task_id=task_name,
        verbose=True,
    )
    print(f"\nSAC mean reward:    {sac_result['mean_reward']:.2f}")
    print(f"Random mean reward: {random_result['mean_reward']:.2f}")
    improvement = sac_result['mean_reward'] - random_result['mean_reward']
    print(f"Improvement:        {improvement:+.2f}")

## 6. Switching to a MARL Algorithm

To use a **multi-agent** library (RLlib, MAPPO, IPPO), use `MARLWrapper` instead.
The wrapper exposes the PettingZoo Parallel API that these libraries expect.

In [ ]:
# PettingZoo Parallel API: obs/actions are dicts keyed by agent id
marl_env = MARLWrapper(make_task_env(task_name, split="train"))

obs, info = marl_env.reset(seed=0)
print("Agent IDs:", list(obs.keys()))
print("Obs shape per agent:", {k: v.shape for k, v in obs.items()})

# Step with individual actions per agent
actions = {agent: marl_env.action_space(agent).sample() for agent in marl_env.agents}
obs, rewards, terminations, truncations, info = marl_env.step(actions)
print("Rewards:", {k: f"{v:.3f}" for k, v in rewards.items()})

print("\n--- To use with RLlib ---")
print("from ray.rllib.env.wrappers.pettingzoo_env import ParallelPettingZooEnv")
print("rllib_env = ParallelPettingZooEnv(MARLWrapper(make_task_env('marl_opf')))")

---

## Summary

| Wrapper | Use when | Key property |
|---|---|---|
| `make_task_env(name)` | Raw access, custom loops | Native PowerZoo API |
| `GymnasiumWrapper(env)` | `evaluate()`, quick testing | `gym.Env` compatible |
| `FlattenWrapper(gym_env)` | SB3 / CleanRL / Tianshou | Flat 1D obs + act |
| `MARLWrapper(env)` | RLlib / MAPPO / IPPO | PettingZoo Parallel API |

## Next

→ [NB03 — Obs / Action / Reward Anatomy](./nb03_obs_action_reward.ipynb) — understand what each observation dimension means